In [19]:
import pandas as pd

In [20]:
df = pd.read_csv("../results.csv", delimiter=";")

In [21]:
df.sort_values("Date")

,Date,Kod,Kurs
0,2017-01-01 12:00:00,ABB,217
1,2017-01-01 12:00:01,NCC,122
2,2017-01-01 12:00:02,ABB,218
3,2017-01-01 12:00:03,NCC,123
4,2017-01-01 12:00:04,NCC,121
5,2017-01-01 12:00:05,AddLife B,21
6,2017-01-01 12:00:06,NCC,121
7,2017-01-01 12:00:06,SSAB B,221
8,2017-01-01 12:01:04,8TRA,226
9,2017-01-01 12:01:05,AddLife B,27


In [22]:
df['Date'] = df['Date'].astype(str).str[:10]

d2 = df.groupby(["Kod","Date"])["Kurs"].last().reset_index()

In [23]:
d2

,Kod,Date,Kurs
0,8TRA,2017-01-01,226
1,8TRA,2017-01-02,225
2,ABB,2017-01-01,219
3,ABB,2017-01-02,222
4,AddLife B,2017-01-01,27
5,AddLife B,2017-01-02,38
6,NCC,2017-01-01,119
7,NCC,2017-01-02,121
8,SSAB B,2017-01-01,221
9,SSAB B,2017-01-02,209


In [24]:
def _find_last_stock_per_day(df: pd.DataFrame) -> pd.DataFrame:
    """Finds latest stock per day at 'closing time' per day"""
    df['Date'] = df['Date'] = pd.to_datetime(df['Date'], format="mixed")
    df = df.sort_values("Date")  # Sorting by date and time
    df["_Day"] = df["Date"].dt.date
    df = df.groupby(["Kod", "_Day"]).last().reset_index()
    return df

def extract_stock_growth(df: pd.DataFrame) -> pd.DataFrame:
    """This function extracts the percent change in growth between groupings on Kod column"""
    last_stocks = _find_last_stock_per_day(df)
    last_stocks["Growth"] = last_stocks.groupby(["Kod"])["Kurs"].pct_change()
    merged = last_stocks.groupby("Kod").agg(
        Date_start=('Date', 'first'),
        Kurs_start=('Kurs', 'first'),
        Date_end=('Date', 'last'),
        Kurs_end=('Kurs', 'last'),
        Growth=('Growth', 'last')
    ).reset_index()
    return merged

In [25]:
d3 = extract_stock_growth(d2)
print(d3)

         Kod Date_start  Kurs_start   Date_end  Kurs_end    Growth
0       8TRA 2017-01-01         226 2017-01-02       225 -0.004425
1        ABB 2017-01-01         219 2017-01-02       222  0.013699
2  AddLife B 2017-01-01          27 2017-01-02        38  0.407407
3        NCC 2017-01-01         119 2017-01-02       121  0.016807
4     SSAB B 2017-01-01         221 2017-01-02       209 -0.054299


In [26]:
type(d3)

pandas.DataFrame

In [27]:
d3

,Kod,Date_start,Kurs_start,Date_end,Kurs_end,Growth
0,8TRA,2017-01-01,226,2017-01-02,225,-0.004425
1,ABB,2017-01-01,219,2017-01-02,222,0.013699
2,AddLife B,2017-01-01,27,2017-01-02,38,0.407407
3,NCC,2017-01-01,119,2017-01-02,121,0.016807
4,SSAB B,2017-01-01,221,2017-01-02,209,-0.054299


In [28]:
def pick_winners(df: pd.DataFrame) -> pd.DataFrame:
    """Creates Winner entries"""
    df_sorted = df.sort_values("Growth", ascending=False).reset_index(drop=True)
    df_sorted["rank"] = df_sorted.index + 1
    return df_sorted

In [29]:
b = pick_winners(d3)
print(b)

         Kod Date_start  Kurs_start   Date_end  Kurs_end    Growth  rank
0  AddLife B 2017-01-01          27 2017-01-02        38  0.407407     1
1        NCC 2017-01-01         119 2017-01-02       121  0.016807     2
2        ABB 2017-01-01         219 2017-01-02       222  0.013699     3
3       8TRA 2017-01-01         226 2017-01-02       225 -0.004425     4
4     SSAB B 2017-01-01         221 2017-01-02       209 -0.054299     5


In [30]:
import pandas as pd
from os import path


def load_csv(path_to_file: str) -> pd.DataFrame:
    """Loads CSV file at specified path and delivers a Pandas Dataframe"""

    if not path.exists(path_to_file):
        raise FileNotFoundError("File does not exist.")

    return pd.read_csv(path_to_file, delimiter=';')

def _find_last_stock_per_day(df: pd.DataFrame) -> pd.DataFrame:
    """Finds latest stock per day at 'closing time' per day"""

    df = df.sort_values("Date") # Sorting by date and time
    df['Date'] = df['Date'].astype(str).str[:10] # Makes date unique, not time. Later: Exclude last by date.
    df = df.groupby(["Kod", "Date"]).last().reset_index() # Here: Last by date.
    return df

def _extract_stock_growth(df: pd.DataFrame) -> pd.DataFrame:
    """Extracts stock growth at 'closing time' per day and creates a new DataFrame
    with aggregate of flat row of data per Kod"""

    last_stocks = _find_last_stock_per_day(df)
    last_stocks["Growth"] = last_stocks.groupby(["Kod"])["Kurs"].pct_change()
    merged = last_stocks.groupby("Kod").agg(
        Date_start=('Date', 'first'),
        Kurs_start=('Kurs', 'first'),
        Date_end=('Date', 'last'),
        Kurs_end=('Kurs', 'last'),
        Growth=('Growth', 'last')
    ).reset_index()
    return merged


def pick_winners(df: pd.DataFrame) -> pd.DataFrame:
    """Creates Winner entries"""

    df = _extract_stock_growth(df)
    df_sorted = df.sort_values("Growth", ascending=False).reset_index(drop=True)
    df_sorted["rank"] = df_sorted.index + 1
    return df_sorted

In [31]:
e = pick_winners(df)

In [32]:
e

,Kod,Date_start,Kurs_start,Date_end,Kurs_end,Growth,rank
0,AddLife B,2017-01-01,21,2017-01-02,38,0.809524,1
1,ABB,2017-01-01,218,2017-01-02,222,0.018349,2
2,8TRA,2017-01-01,226,2017-01-02,225,-0.004425,3
3,NCC,2017-01-01,122,2017-01-02,121,-0.008197,4
4,SSAB B,2017-01-01,221,2017-01-02,209,-0.054299,5
